#### Task: get the transformer to respond to "What is love?" and "Love is what?" Answer: "Awesome"

In [1]:
# imports

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

import lightning as L
from torch.profiler import profile, schedule
from lightning.pytorch.profilers import PyTorchProfiler

/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <0B7EB158-53DC-3403-8A49-22178CAB4612> /Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/torchvision/image.so
  Reason: tried: '/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/aryamantepal/anaconda3/envs/torch-lightning/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan o

In [2]:
mps_device = torch.device("mps")

In [3]:
# # data

# token_to_id = {
#     'what': 0,
#     'is': 1,
#     'love': 2,
#     'awesome': 3,
#     '<EOS>': 4,
# }

# id_to_token = dict(map(reversed, token_to_id.items()))

# inputs = torch.tensor([[token_to_id['what'],
#                       token_to_id['is'],
#                       token_to_id['love'],
#                       token_to_id['<EOS>'],
#                       token_to_id['awesome']],
                       
#                        [token_to_id['love'],
#                       token_to_id['is'],
#                       token_to_id['what'],
#                       token_to_id['<EOS>'],
#                       token_to_id['awesome']]])

# labels = torch.tensor([[token_to_id['is'],
#                       token_to_id['love'],
#                       token_to_id['<EOS>'],
#                       token_to_id['awesome'],
#                       token_to_id['<EOS>']],
                       
#                       [token_to_id['is'],
#                        token_to_id['what'],
#                       token_to_id['<EOS>'],
#                       token_to_id['awesome'],
#                       token_to_id['<EOS>']]])

# dataset = TensorDataset(inputs, labels)
# dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# mps_device = torch.device("mps")

In [4]:
class PositionEncoding(nn.Module):
    
    def __init__(self, d_model=2, max_len=6):
        
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)
        embedding_index = torch.arange(start=0, end=d_model, step=2).float()
        
        div_term = 1/torch.tensor(10000.0)**(embedding_index / d_model)
        

        pe[:, 0::2] = torch.sin(position * div_term) 
        pe[:, 1::2] = torch.cos(position * div_term) 
        
        self.register_buffer('pe', pe) 

        
    def forward(self, word_embeddings):
        T = word_embeddings.size(1)
        return word_embeddings + self.pe[:T, :].unsqueeze(0)

In [5]:
# # query, key, value for each token
# class Attention(nn.Module): 
    
#     def __init__(self, d_model=2):
        
#         super().__init__()
        
#         self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
#         self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
#         self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

#     def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):
#         """Calculate Masked Self-Attention for each token; encodings can come from diff sources, allowing Encoder-Decoder Attention"""
#         # encodings: (B, T, D)
#         q = self.W_q(encodings_for_q)
#         k = self.W_k(encodings_for_k)
#         v = self.W_v(encodings_for_v)

#         # (B, T, T)
#         sims = torch.matmul(q, k.transpose(-2, -1))

#         d_k = k.size(-1)
#         scaled_sims = sims / (d_k ** 0.5)

#         if mask is not None:
#             # mask: (T, T) -> (1, T, T) -> broadcast
#             scaled_sims = scaled_sims.masked_fill(mask.unsqueeze(0), -1e9)

#         attention_probs = F.softmax(scaled_sims, dim=-1)
#         attention_scores = torch.matmul(attention_probs, v)  # (B, T, D)

#         return attention_scores

In [6]:
# class DecoderOnlyTransformer(L.LightningModule):
    
#     def __init__(self, num_tokens=4, d_model=2, max_len=6):
#         # num_tokens: number of tokens in the vocabulary
#         # d_model: number of values used to represent each token
#         # max_len: maximum length of the input + output
#         super().__init__()
        
#         L.seed_everything(seed=42)
        
#         self.we = nn.Embedding(num_embeddings=num_tokens, 
#                                embedding_dim=d_model)     
#         self.pe = PositionEncoding(d_model=d_model, 
#                                    max_len=max_len)
#         self.self_attention = Attention(d_model=d_model)
#         self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens)
        
#         self.loss = nn.CrossEntropyLoss()
        
        
#     def forward(self, token_ids):
#         # token_ids: (B, T)
#         word_embeddings = self.we(token_ids)        # (B, T, D)
#         position_encoded = self.pe(word_embeddings)
        
#         # mask prevents early tokens looking at late tokens when calculating attention
#         B, T = token_ids.shape
#         mask = torch.tril(
#             torch.ones((T, T), device=token_ids.device)
#         ).bool()
#         mask = ~mask
        
#         self_attention_values = self.self_attention(position_encoded, 
#                                                     position_encoded, 
#                                                     position_encoded, 
#                                                     mask=mask)
                
#         residual = position_encoded + self_attention_values       
#         fc_layer_output = self.fc_layer(residual)
        
#         return fc_layer_output
    
#     def configure_optimizers(self): 
#         return Adam(self.parameters(), lr=0.1)
    
    
#     def training_step(self, batch, batch_idx): 
#         input_tokens, labels = batch  # (B, T)
#         logits = self.forward(input_tokens)  # (B, T, V)
#         loss = self.loss(
#         logits.view(-1, logits.size(-1)),  # (B*T, V)
#         labels.view(-1)                    # (B*T)
#         )           
#         return loss

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # Project then reshape into heads: (B, T, D) -> (B, H, T, head_dim)
        q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # (B, H, T, T) — this is the matrix that explodes in size with seq_len
        sims = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask is (T, T), broadcast to (1, 1, T, T) for all batches and heads
            sims = sims.masked_fill(mask.unsqueeze(0).unsqueeze(0), -1e9)

        attn_probs = F.softmax(sims, dim=-1)
        attn_out = torch.matmul(attn_probs, v)  # (B, H, T, head_dim)

        # Recombine heads: (B, H, T, head_dim) -> (B, T, D)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(attn_out)


class DecoderBlock(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        # d_ff = 4 * d_model is standard — this is where most FLOPs live
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, mask=None):
        # Pre-norm style (more stable than post-norm at scale)
        x = x + self.attn(self.norm1(x), mask=mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyTransformer(L.LightningModule):
    def __init__(self, num_tokens=32000, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, max_len=512):
        super().__init__()
        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_tokens, d_model)
        self.pe = PositionEncoding(d_model=d_model, max_len=max_len)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.fc_layer = nn.Linear(d_model, num_tokens)
        self.loss = nn.CrossEntropyLoss()

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.we(token_ids)       # (B, T, D)
        x = self.pe(x)

        mask = ~torch.tril(torch.ones(T, T, device=token_ids.device)).bool()

        for layer in self.layers:
            x = layer(x, mask=mask)

        x = self.norm(x)
        return self.fc_layer(x)      # (B, T, V)

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=3e-4)

    def training_step(self, batch, batch_idx):
        input_tokens, labels = batch
        logits = self.forward(input_tokens)
        loss = self.loss(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss

In [8]:
# Synthetic dataset to match your existing dataloader pattern
num_tokens = 32000
seq_len = 512
batch_size = 16
num_samples = 256  # was 64, gives you 16 batches instead of 4

input_data = torch.randint(0, num_tokens, (num_samples, seq_len))
label_data = torch.randint(0, num_tokens, (num_samples, seq_len))

dataset = TensorDataset(input_data, label_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Model
device = mps_device
model = DecoderOnlyTransformer(
    num_tokens=num_tokens,
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
    max_len=512,
).to(device)

# Train with profiler callback
profiler = PyTorchProfiler(
    schedule=schedule(wait=1, warmup=2, active=3, repeat=1),
    record_shapes=True,
    profile_memory=True,
)

trainer = L.Trainer(
    max_epochs=1,
    accelerator="mps",
    devices=1,
    profiler=profiler,
    enable_checkpointing=False,  # kills the 18% checkpoint overhead
    logger=False,                # removes logging overhead too
)

trainer.fit(model, dataloader)

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name     | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | we       | Embedding        | 16.4 M | train | 0    
1 | pe       | PositionEncoding | 0      | train | 0    
2 | layers   | ModuleList       | 18.9 M | train | 0    
3 | norm     | LayerNorm        | 1.0 K  | train | 0    
4 | fc_layer | Linear           | 16.4 M | train | 0    
5 | loss     | CrossEntropyLoss | 0      | train | 0    
--------------------------------------------------------------
51.7 M    Trainable params
0         Non-trainable params
51.7 M    Total params
206.812   Total estimated model params size (MB)
78        Modules in train mode
0         Modules in eval mode
0         Total Flops
/Users/aryamantepal/anaconda3/envs/torch-lightning/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' do

Epoch 0:  12%|█▎        | 2/16 [00:03<00:24,  0.57it/s]

[W201 18:18:55.593926000 CPUAllocator.cpp:249] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Epoch 0:  31%|███▏      | 5/16 [00:08<00:18,  0.58it/s]

[W201 18:19:00.696334000 collection.cpp:634] Warning: Optimizer.step#Adam.step (function operator())


Epoch 0: 100%|██████████| 16/16 [00:26<00:00,  0.60it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 16/16 [00:26<00:00,  0.60it/s]


FIT Profiler Report
Profile stats for: records
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.85%      77.532ms        55.65%        5.067s        1.689s          48 b    -383.88 Kb             3  
                        [pl][profile]run_training_batch         0.01%       1.168ms        36.83%        3.354s        1.118s          48 b           0 b             3  
[pl][profile][LightningModule]DecoderOnlyTransformer...         0.00%     203.875us        36.82%      